In [1]:
# %% [markdown]
# # Water block: dry episode -> reservoirs -> official drought status (Catalonia)
# **Question:** how many months pass between a rainfall deficit (XEMA) and its reflection in the reservoirs (ACA)
# and in the official drought status (ACA)?
# Output: one figure, one episode table, and a monthly CSV that other blocks (fires, heat) can reuse.

# %% 0. Setup
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})

API = "https://analisi.transparenciacatalunya.cat/api/v3/views/{}/query.json"
load = lambda ds: download_socrata(API.format(ds)).rename(columns=str.upper)

def pick(df, *names):
    for n in names:
        if n in df.columns:
            return n
    raise KeyError(f"None of {names} found. Columns: {list(df.columns)}")

def find(df, rx):
    """First column matching regex, ignoring code columns (we want names)."""
    cols = [c for c in df.columns if re.search(rx, c, re.I) and "CODI" not in c.upper()]
    if not cols:
        raise KeyError(f"No column matches '{rx}'. Columns: {list(df.columns)}")
    return cols[0]

def months_between(a, b):
    return (b.year - a.year) * 12 + b.month - a.month

UNIT_KEY = "ter-llobregat"   # reference exploitation unit (Barcelona area; matches the reservoir system)
MANUAL = {}                  # e.g. {"ppt": 35} if the automatic precipitation variable is wrong
# Comarques draining to the Internal Basins (edit if your team defines the catchment differently)
CATCHMENT = ["Ripollès", "Osona", "Berguedà", "Bages", "Garrotxa", "Selva", "Gironès", "Anoia", "Moianès",
             "Vallès Occidental", "Vallès Oriental", "Maresme", "Baix Llobregat", "Alt Penedès", "Garraf",
             "Barcelonès", "Alt Empordà", "Baix Empordà", "Pla de l'Estany", "Solsonès"]

# %% 1. XEMA daily -> monthly regional SPI-6
path_xema = r"C:\Users\nmayo\OneDrive\Escritorio\Máster - UB\Análisis de datos masivos\Datasets - Agua\Dades_meteorològiques_diàries_de_la_XEMA_20261116.csv"
df_xema = pd.read_csv(path_xema, low_memory=False)
print("Dimensions:", df_xema.shape)
display(df_xema.head())

d = df_xema.rename(columns=str.upper)
stations, variables = load("yqwd-vj5e"), load("4fb2-n3yi")
c_st, c_dt, c_var, c_val = pick(d, "CODI_ESTACIO", "CODI_ESTAT"), pick(d, "DATA_LECTURA", "DATA"), pick(d, "CODI_VARIABLE"), pick(d, "VALOR")
d[c_var] = pd.to_numeric(d[c_var], errors="coerce")
variables["CODI_VARIABLE"] = pd.to_numeric(variables["CODI_VARIABLE"], errors="coerce")

# Precipitation variable code, found through the metadata (check the printout once)
counts = d[c_var].value_counts()
present = (variables[variables.CODI_VARIABLE.isin(counts.index)]
           .assign(n=lambda x: x.CODI_VARIABLE.map(counts)).sort_values("n", ascending=False))
display(present[[c for c in ("CODI_VARIABLE", "NOM_VARIABLE", "ACRONIM", "n") if c in present.columns]].head(15))
if "ppt" in MANUAL:
    ppt_code = MANUAL["ppt"]
else:
    cand = present[present.NOM_VARIABLE.str.contains(r"precipitaci", case=False)]
    if cand.empty:
        raise KeyError("No precipitation variable found. Check the table above and set MANUAL = {'ppt': <code>}")
    ppt_code = cand.iloc[0].CODI_VARIABLE
print("Precipitation variable ->", ppt_code)

daily = d[d[c_var] == ppt_code].assign(
    station=lambda x: x[c_st],
    date=lambda x: pd.to_datetime(x[c_dt], utc=True).dt.tz_localize(None).dt.normalize(),
    ppt=lambda x: pd.to_numeric(x[c_val], errors="coerce"))[["station", "date", "ppt"]]
daily.loc[daily.ppt < 0, "ppt"] = np.nan

# Monthly totals per station (partial months scaled to the full month; months with <25 days discarded)
daily["month"] = daily.date.dt.to_period("M").dt.to_timestamp()
mon = daily.groupby(["station", "month"]).ppt.agg(avg="mean", n="count").reset_index()
mon["ppt"] = (mon.avg * mon.month.dt.days_in_month).where(mon.n >= 25)
P = mon.pivot(index="month", columns="station", values="ppt").asfreq("MS")
P = P.loc[:, P.count() >= 120]                                   # stations with >= 10 years of data

s_id, s_com = pick(stations, "CODI_ESTACIO", "CODI"), next((c for c in ("NOM_COMARCA", "COMARCA") if c in stations.columns), None)
keep = set(stations[stations[s_com].isin(CATCHMENT)][s_id].astype(str)) if s_com else set(stations[s_id].astype(str))
P = P[[c for c in P.columns if str(c) in keep]]

# SPI-6: each station's 6-month accumulated rainfall as a ratio to its own normal for the same window
# (ratio of sums, not mean of monthly ratios), averaged over stations, then standardised over time.
acc6 = P.rolling(6, min_periods=6).sum()
ratio = acc6 / acc6.groupby(acc6.index.month).transform("mean")
reg = ratio.mean(axis=1).where(ratio.notna().sum(axis=1) >= 10)
spi6 = ((reg - reg.mean()) / reg.std()).rename("spi6")           # SPI-like index (no gamma fit)
print("Stations in catchment:", P.shape[1])

# %% 2. Reservoirs (ACA): system storage in % of capacity
res = load("gn9e-3qhr")
c_day, c_est = pick(res, "DIA", "DATA"), pick(res, "ESTACIO", "ESTACI", "EMBASSAMENT")
c_vol, c_pct = pick(res, "VOLUM_EMBASSAT"), pick(res, "PERCENTATGE_VOLUM_EMBASSAT")
res[c_day] = pd.to_datetime(res[c_day]).dt.normalize()
for c in (c_vol, c_pct):
    res[c] = pd.to_numeric(res[c], errors="coerce")
SYSTEM = r"\bsau\b|susqueda|baells|llosa del cavall|sant pon"   # reservoirs of the Ter-Llobregat unit
sel = res[c_est].astype(str).str.contains(SYSTEM, case=False, regex=True)
print("Included:", sorted(res[sel][c_est].unique()), "\nExcluded:", sorted(res[~sel][c_est].unique()))
if sel.any():
    res = res[sel]
else:
    print("WARNING: no reservoir matched SYSTEM, using all Internal Basins reservoirs")
res["cap"] = (res[c_vol] / (res[c_pct] / 100)).groupby(res[c_est]).transform("median")
sysd = res.groupby(c_day).agg(vol=(c_vol, "sum"), cap=("cap", "sum"), n=(c_est, "nunique"))
sysd = sysd[sysd.n >= 0.9 * sysd.n.max()]                       # drop days with missing reservoirs
resm = (100 * sysd.vol / sysd.cap).resample("MS").mean().rename("res")

# %% 3. Official drought status (ACA): change log -> monthly state of the reference unit
sq = load("i5n8-43cw")
print("Dimensions:", sq.shape); display(sq.head())
c_date, c_unit = find(sq, r"^data"), find(sq, r"unitat")
c_hyd, c_plu = find(sq, r"estat.*hidrol"), find(sq, r"estat.*pluvi")
print("Using ->", {"date": c_date, "unit": c_unit, "hydrological": c_hyd, "pluviometric": c_plu})
print("Hydrological values :", sq[c_hyd].dropna().unique()[:12])
print("Pluviometric values :", sq[c_plu].dropna().unique()[:12])

def lvl_hyd(x):
    """0 normal, 1 pre-alert, 2 alert, 3 exceptionality, 4/5/6 emergency I/II/III."""
    x = str(x).lower()
    if "prealerta" in x: return 1
    if "alerta" in x: return 2
    if "excepcional" in x: return 3
    if "emerg" in x:
        m = re.search(r"emerg\w*\s*(iii|ii|i)\b", x)
        return {"i": 4, "ii": 5, "iii": 6}[m.group(1) if m else "i"]
    return 0 if "normal" in x else np.nan

def lvl_plu(x):
    """0 normal, 1 moderate, 2 severe, 3 extreme."""
    x = str(x).lower()
    for k, v in (("extrem", 3), ("sever", 2), ("moderad", 1), ("normal", 0)):
        if k in x: return v
    return np.nan

sq["date"] = pd.to_datetime(sq[c_date], utc=True).dt.tz_localize(None)
ref = sq[sq[c_unit].astype(str).str.lower().str.replace(" ", "-").str.contains(UNIT_KEY)].sort_values("date")
if ref.empty:
    raise ValueError(f"No unit matches '{UNIT_KEY}'. Units: {sq[c_unit].unique()}")
ref = ref.assign(hyd=ref[c_hyd].map(lvl_hyd), plu=ref[c_plu].map(lvl_plu))
off = ref.set_index("date")[["hyd", "plu"]].resample("MS").last()
off = off.reindex(pd.date_range(off.index.min(), max(off.index.max(), resm.index.max()), freq="MS")).ffill()  # change log -> state in force each month
print("Official series:", f"{off.index.min():%Y-%m} -> {off.index.max():%Y-%m}")

# %% 4. Monthly table (shareable with the fires / heat blocks)
monthly = pd.concat([spi6, resm, off], axis=1).loc["2005-01":]
monthly.index.name = "month"
monthly.to_csv("water_monthly.csv")
display(monthly.dropna().tail())

# %% 5. Dry episodes (SPI-6 <= -1 for 3 consecutive months) and lags to each milestone
run = (monthly.spi6 <= -1).astype(int).rolling(3).sum().eq(3)
starts = [t - pd.DateOffset(months=2) for t in monthly.index[run & ~run.shift(1, fill_value=False)]]
onsets = []
for t in starts:
    if not onsets or (t - onsets[-1]).days > 540:                # merge runs of the same episode
        onsets.append(t)

def lag(col, o, cond):
    """Months from onset o until the series first meets cond (within 5 years); NaN if never / no data yet."""
    s = monthly[col].dropna()
    if s.empty or o < s.index.min():
        return np.nan
    w = s.loc[o:o + pd.DateOffset(months=60)]
    w = w[cond(w)]
    return months_between(o, w.index[0]) if len(w) else np.nan

rows = []
for o in onsets:
    w = monthly.res.loc[o:o + pd.DateOffset(months=60)]
    rows.append({"Dry onset": f"{o:%Y-%m}",
                 "Min reservoir %": round(w.min(), 1) if w.notna().any() else np.nan,
                 "Reservoir % at onset": round(monthly.res.get(o, np.nan), 1),
                 "Reservoirs <40% [m]": lag("res", o, lambda s: s < 40),
                 "Official rain: severe [m]": lag("plu", o, lambda s: s >= 2),
                 "Official hydro: alert [m]": lag("hyd", o, lambda s: s >= 2),
                 "Official hydro: exceptionality [m]": lag("hyd", o, lambda s: s >= 3),
                 "Official hydro: emergency [m]": lag("hyd", o, lambda s: s >= 4)})
episodes = pd.DataFrame(rows)
display(episodes)

# %% 6. Figure: rainfall deficit -> reservoirs, each with its official status in the background
HYD = {0: ("Normal", "#ffffff"), 1: ("Pre-alert", "#e0e0e0"), 2: ("Alert", "#fddbc7"), 3: ("Exceptionality", "#ef8a62"),
       4: ("Emergency I", "#b2182b"), 5: ("Emergency II", "#67001f"), 6: ("Emergency III", "#3d0010")}
PLU = {0: ("Normal", "#ffffff"), 1: ("Moderate", "#fddbc7"), 2: ("Severe", "#ef8a62"), 3: ("Extreme", "#b2182b")}

def shade(ax, s, pal, title):
    s = s.dropna()
    for _, g in s.groupby((s != s.shift()).cumsum()):
        ax.axvspan(g.index[0], g.index[-1] + pd.DateOffset(months=1), color=pal[int(g.iloc[0])][1], alpha=.5, lw=0)
    ax.legend(handles=[Patch(color=pal[i][1], alpha=.5, label=pal[i][0]) for i in sorted(s.unique().astype(int))],
              title=title, loc="upper left", fontsize=8, ncol=2)

fig, ax = plt.subplots(2, 1, figsize=(13, 8), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]})
shade(ax[0], monthly.plu, PLU, "Official rainfall status")
ax[0].plot(monthly.index, monthly.spi6, color="#2166ac", lw=1.6)
ax[0].axhline(-1, ls="--", color="k", lw=.8); ax[0].set_ylabel("SPI-6 (XEMA)")
ax[0].set_title("Rainfall deficit  ->  reservoirs  ->  official drought status (Ter-Llobregat)")
shade(ax[1], monthly.hyd, HYD, "Official hydrological status")
ax[1].plot(monthly.index, monthly.res, color="#08519c", lw=2); ax[1].set_ylim(0, 100); ax[1].set_ylabel("Reservoirs (% capacity)")
for o in onsets:
    for a in ax:
        a.axvline(o, color="#d6604d", ls="--", alpha=.7)
ax[1].set_xlim(monthly.res.first_valid_index(), monthly.index.max())
plt.tight_layout(); plt.savefig("fig_water_chain.png", dpi=200); plt.show()

Dimensions: (33475032, 10)


,ID,CODI_ESTACIO,NOM_ESTACIO,DATA_LECTURA,CODI_VARIABLE,NOM_VARIABLE,VALOR,UNITAT,HORA _TU,Estat
0,VK1000098801,VK,Raimat,01/09/1988,1.000,Temperatura mitjana diària,"23,3",°C,NaN,Representatiu
1,VK1002098801,VK,Raimat,01/09/1988,1.002,Temperatura mínima diària + hora,"18,3",°C,0:00,Representatiu
2,VK1100098801,VK,Raimat,01/09/1988,1.100,Humitat relativa mitjana diària,58,%,NaN,Representatiu
3,VK1300098801,VK,Raimat,01/09/1988,1.300,Precipitació acumulada diària,"0,6",mm,NaN,Representatiu
4,VK1301098801,VK,Raimat,01/09/1988,1.301,Precipitació acumulada diària (8-8 h),"0,6",mm,NaN,Representatiu


NameError: name 'download_socrata' is not defined